# Week 1: Data Profiling, Cleaning & Validation
**Project:** Heavy Supplier, Inventory & Warehouse Analytics (CadetX)

**How to use:** put the 12 CSV files in a folder called `data/` next to this notebook, then run all cells.
It creates an `outputs/` folder with the profiling summary, column profile, cleaning log, validation report and cleaned CSV files.

In [1]:
import pandas as pd, numpy as np, glob, os
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50)

DATA_DIR, OUT_DIR = 'data', 'outputs'
os.makedirs(f'{OUT_DIR}/cleaned', exist_ok=True)

dfs = {os.path.basename(f)[:-4]: pd.read_csv(f, low_memory=False) for f in sorted(glob.glob(f'{DATA_DIR}/*.csv'))}
print(len(dfs), 'tables loaded:', ', '.join(dfs))

12 tables loaded: branches, customers, inventory_master, invoices, payments, products, purchase_orders_header, purchase_orders_lines, sales_orders_header, sales_orders_lines, stock_ledger, suppliers


## 1. Profiling: one row per table

In [2]:
summary = pd.DataFrame([{
    'table': n, 'rows': len(d), 'columns': d.shape[1],
    'null_cells': int(d.isna().sum().sum()), 'duplicate_rows': int(d.duplicated().sum())
} for n, d in dfs.items()])
summary.to_csv(f'{OUT_DIR}/profiling_summary.csv', index=False)
summary

,table,rows,columns,null_cells,duplicate_rows
0,branches,6,13,0,0
1,customers,500,15,0,0
2,inventory_master,180,8,0,0
3,invoices,18033,10,0,0
4,payments,19257,5,0,0
5,products,30,23,0,0
6,purchase_orders_header,24000,10,2370,0
7,purchase_orders_lines,155495,9,0,0
8,sales_orders_header,20000,11,0,0
9,sales_orders_lines,130402,9,0,0


## 2. Profiling: one row per column (use this to build the data dictionary)

In [3]:
rows = []
for n, d in dfs.items():
    for c in d.columns:
        s = d[c]
        rows.append({'table': n, 'column': c, 'dtype': str(s.dtype), 'nulls': int(s.isna().sum()),
                     'unique_values': int(s.nunique()), 'min': str(s.min()), 'max': str(s.max())})
col_profile = pd.DataFrame(rows)
col_profile.to_csv(f'{OUT_DIR}/column_profile.csv', index=False)
col_profile.head(20)

,table,column,dtype,nulls,unique_values,min,max
0,branches,branch_id,str,0,6,AHM001,PUN001
1,branches,branch_name,str,0,6,Ahmedabad West Hub,Pune Distribution
2,branches,city,str,0,6,Ahmedabad,Pune
3,branches,state,str,0,6,Delhi,West Bengal
4,branches,region,str,0,4,East,West
5,branches,warehouse_type,str,0,4,Branch,Service Hub
6,branches,warehouse_capacity,str,0,6,29880 sqft,45230 sqft
7,branches,service_center_available,str,0,2,No,Yes
8,branches,manager_id,int64,0,6,4390,9593
9,branches,total_employees,int64,0,6,49,92


## 3. Cleaning (every change is written to the cleaning log)

In [4]:
log = []
def note(table, column, action, records): log.append({'table': table, 'column': column, 'action': action, 'records_affected': int(records)})

for n, d in dfs.items():
    for c in d.columns:
        # trim stray spaces in text columns
        if d[c].dtype == object:
            trimmed = d[c].str.strip()
            changed = (d[c].notna() & (trimmed != d[c])).sum()
            if changed:
                d[c] = trimmed; note(n, c, 'Trimmed leading/trailing spaces', changed)
        # convert text dates to real dates
        if c.endswith('_date') or c == 'customer_since':
            before = d[c].notna().sum()
            d[c] = pd.to_datetime(d[c], errors='coerce')
            note(n, c, 'Converted text to datetime', before)
            bad = before - d[c].notna().sum()
            if bad: note(n, c, 'Could not be parsed, set to blank', bad)

# warehouse capacity is text like '45230 sqft' -> make a numeric column
b = dfs['branches']
b['warehouse_capacity_sqft'] = b['warehouse_capacity'].str.extract(r'(\d+)')[0].astype(int)
note('branches', 'warehouse_capacity', 'Added numeric column warehouse_capacity_sqft', len(b))

# blank received_date is expected for cancelled purchase orders
po = dfs['purchase_orders_header']
note('purchase_orders_header', 'received_date', 'Blank dates left as is (see check below)', po['received_date'].isna().sum())

cleaning_log = pd.DataFrame(log)
cleaning_log.to_csv(f'{OUT_DIR}/cleaning_log.csv', index=False)
for n, d in dfs.items(): d.to_csv(f'{OUT_DIR}/cleaned/{n}.csv', index=False)
cleaning_log

,table,column,action,records_affected
0,customers,customer_since,Converted text to datetime,500
1,customers,last_purchase_date,Converted text to datetime,500
2,invoices,invoice_date,Converted text to datetime,18033
3,invoices,due_date,Converted text to datetime,18033
4,payments,payment_date,Converted text to datetime,19257
5,products,last_purchase_date,Converted text to datetime,30
6,purchase_orders_header,order_date,Converted text to datetime,24000
7,purchase_orders_header,expected_delivery_date,Converted text to datetime,24000
8,purchase_orders_header,received_date,Converted text to datetime,21630
9,sales_orders_header,order_date,Converted text to datetime,20000


## 4. Validation checks
`REVIEW` means the team should look at the records and decide (fix, exclude, or explain).

In [5]:
so, sl = dfs['sales_orders_header'], dfs['sales_orders_lines']
po, pl = dfs['purchase_orders_header'], dfs['purchase_orders_lines']
inv, pay = dfs['invoices'], dfs['payments']
prod, cust, br, sup = dfs['products'], dfs['customers'], dfs['branches'], dfs['suppliers']
led, invm = dfs['stock_ledger'], dfs['inventory_master']

checks = []
def check(name, bad, total):
    checks.append({'check': name, 'records_checked': int(total), 'issues': int(bad), 'result': 'PASS' if bad == 0 else 'REVIEW'})

# totals
m = so.set_index('so_id')['total_order_value']; s = sl.groupby('so_id')['line_total'].sum().reindex(m.index)
check('Sales order total = sum of its lines', ((m - s).abs() > 1).sum(), len(m))
m = po.set_index('po_id')['total_cost']; s = pl.groupby('po_id')['line_total'].sum().reindex(m.index)
check('Purchase order total = sum of its lines', ((m - s).abs() > 1).sum(), len(m))
x = inv.merge(so[['so_id', 'grand_total']], on='so_id', suffixes=('', '_so'))
check('Invoice grand total = sales order grand total', ((x.grand_total - x.grand_total_so).abs() > 1).sum(), len(x))
check('Only one invoice per sales order', inv['so_id'].duplicated().sum(), len(inv))
check('invoice_id is unique (no ID reused)', inv['invoice_id'].duplicated().sum(), len(inv))
# invoice IDs that appear more than once cannot be matched safely to payments, so payment checks skip them
inv_ok = inv[~inv['invoice_id'].duplicated(keep=False)]

# payments vs invoice status
paid = pay.groupby('invoice_id')['payment_amount'].sum()
i = inv_ok.assign(paid=inv_ok['invoice_id'].map(paid).fillna(0))
check("'Paid' invoices fully paid", ((i.payment_status == 'Paid') & (i.paid < i.grand_total - 1)).sum(), (i.payment_status == 'Paid').sum())
check("'Unpaid' invoices have no payments", ((i.payment_status == 'Unpaid') & (i.paid > 0)).sum(), (i.payment_status == 'Unpaid').sum())
check("'Partially Paid' invoices paid in part", ((i.payment_status == 'Partially Paid') & ~((i.paid > 0) & (i.paid < i.grand_total))).sum(), (i.payment_status == 'Partially Paid').sum())

# dates
check('Sales delivery date is not before order date', (so.delivery_date < so.order_date).sum(), len(so))
r = po[po.received_date.notna()]
check('PO received date is not before order date', (r.received_date < r.order_date).sum(), len(r))
check('Blank received_date only on cancelled POs', (po.received_date.isna() & (po.po_status != 'Cancelled')).sum(), po.received_date.isna().sum())
y = pay.merge(inv_ok[['invoice_id', 'invoice_date']], on='invoice_id')
check('Payment date is not before invoice date', (y.payment_date < y.invoice_date).sum(), len(y))

# keys exist in parent tables
def fk(name, child, parent): check(f'{name} exists in parent table', (~child.isin(parent)).sum(), len(child))
fk('sales_orders_lines.product_id', sl.product_id, prod.product_id)
fk('purchase_orders_lines.product_id', pl.product_id, prod.product_id)
fk('sales_orders_header.customer_id', so.customer_id, cust.customer_id)
fk('sales_orders_header.branch_id', so.branch_id, br.branch_id)
fk('purchase_orders_header.supplier_id', po.supplier_id, sup.supplier_id)
fk('stock_ledger.product_id', led.product_id, prod.product_id)
fk('stock_ledger.branch_id', led.branch_id, br.branch_id)

# stock ledger vs inventory master
last = led.sort_values('movement_date', kind='stable').groupby(['product_id', 'branch_id'])['running_balance'].last()
z = invm.set_index(['product_id', 'branch_id'])['current_stock']
check('Last ledger balance = inventory_master current_stock', (last.reindex(z.index) != z).sum(), len(z))
check('No negative running balances in stock ledger', (led.running_balance < 0).sum(), len(led))

validation = pd.DataFrame(checks)
validation.to_csv(f'{OUT_DIR}/validation_report.csv', index=False)
validation

,check,records_checked,issues,result
0,Sales order total = sum of its lines,20000,0,PASS
1,Purchase order total = sum of its lines,24000,0,PASS
2,Invoice grand total = sales order grand total,18033,0,PASS
3,Only one invoice per sales order,18033,0,PASS
4,invoice_id is unique (no ID reused),18033,197,REVIEW
5,'Paid' invoices fully paid,12264,0,PASS
6,'Unpaid' invoices have no payments,1798,0,PASS
7,'Partially Paid' invoices paid in part,3578,0,PASS
8,Sales delivery date is not before order date,20000,0,PASS
9,PO received date is not before order date,21630,0,PASS


## 5. Findings and decisions (Week 1)

**REVIEW checks that matter:**
1. 197 `invoice_id` values are reused across different sales orders (not a missing value — a duplicate value). This means a payment referencing one of these invoice numbers can't be matched with certainty to the correct sales order.
2. 14 of 180 product-branch pairs: the last `stock_ledger` running balance does not match `inventory_master.current_stock`.

**Team decision (agreed Week 1):**
- Issue 1: Exclude these 197 invoices from payment-related analysis (e.g. who has/hasn't paid) until further notice. Documented here and in the Week 1 sprint notes.
- Issue 2: Flagged as a data quality issue to investigate further in Week 2, not yet resolved.

**KPIs and analysis questions proposed this week:** see `kpi_proposal.md` in this folder.